# Evaluation Pipeline for Portfolio AI Agent

This notebook evaluates the Portfolio AI Agent along two dimensions:

| Section | What it measures |
|---|---|
| **Retrieval Evaluation** | Does the search engine surface the right document? (Hit Rate @5, MRR) |
| **LLM-as-Judge** | Is the agent's answer relevant, accurate, language-matched, and source-cited? |

**Prerequisites:**
- `eval/ground_truth.csv` must exist (run `data-gen.ipynb` first)
- `GROQ_API_KEY` must be set in your environment

In [ ]:
import sys
import os
import json
import time
import asyncio
import pandas as pd
from collections import defaultdict
from tqdm.auto import tqdm

# ── Add parent directory to sys.path ──────────────────────────────────────────
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
PARENT_DIR = os.path.dirname(NOTEBOOK_DIR)
if PARENT_DIR not in sys.path:
    sys.path.insert(0, PARENT_DIR)

print(f"Parent dir on sys.path: {PARENT_DIR}")
print(f"Python: {sys.version}")

## 1. Load Ground Truth

Load the CSV generated by `data-gen.ipynb`.

In [ ]:
GROUND_TRUTH_PATH = os.path.join(NOTEBOOK_DIR, "ground_truth.csv")

if not os.path.exists(GROUND_TRUTH_PATH):
    raise FileNotFoundError(
        f"ground_truth.csv not found at {GROUND_TRUTH_PATH}.\n"
        "Please run data-gen.ipynb first."
    )

ground_truth = pd.read_csv(GROUND_TRUTH_PATH)

print(f"Loaded {len(ground_truth)} ground truth pairs")
print(f"Languages: {ground_truth['language'].value_counts().to_dict()}")
print(f"\nSample rows:")
ground_truth.head(4)

## 2. Initialise the Search Tool and Agent

We build the index **with chunking** (the default production setting) so the retrieval evaluation
reflects real-world performance.

In [ ]:
from ingest import read_repo_data
from search_tools import SearchTool
from search_agent import init_agent, run_agent

# Build index with chunking (production mode)
index, records = read_repo_data(chunk=True)

# Instantiate the search tool
search_tool = SearchTool(index=index, records=records)

# Instantiate the portfolio agent
agent = init_agent(search_tool)

print("Search tool and agent ready.")
print(f"Index contains {len(records)} chunks")

---
# Section A: Retrieval Evaluation

For each question we run `search_tool.search(question)` and check whether the **expected `doc_id`**
appears in the top-5 results.

**Metrics:**
- **Hit Rate @5** — fraction of questions where the correct doc appears in the top 5 results
- **MRR** (Mean Reciprocal Rank) — average of 1/rank for the first correct result (0 if not found in top 5)

In [ ]:
def evaluate_retrieval(ground_truth_df: pd.DataFrame, tool: SearchTool, top_k: int = 5) -> pd.DataFrame:
    """
    Run retrieval for every question in ground_truth_df.

    Returns a DataFrame with one row per question, containing:
        question_id, question, language, doc_id,
        hit (bool), rank (int, 0 = not found)
    """
    rows = []

    for _, row in tqdm(ground_truth_df.iterrows(), total=len(ground_truth_df), desc="Retrieval eval"):
        results = tool.search(row["question"])

        # Check whether the expected doc_id appears in the top-k results
        hit = False
        rank = 0
        for i, result in enumerate(results[:top_k], start=1):
            if result.get("doc_id") == row["doc_id"]:
                hit = True
                rank = i
                break

        rows.append(
            {
                "question_id": row["question_id"],
                "question": row["question"],
                "language": row["language"],
                "doc_id": row["doc_id"],
                "hit": hit,
                "rank": rank,
            }
        )

    return pd.DataFrame(rows)


retrieval_results = evaluate_retrieval(ground_truth, search_tool, top_k=5)
print(f"Retrieval evaluation complete: {len(retrieval_results)} questions evaluated")

In [ ]:
# ── Compute Hit Rate @5 and MRR ───────────────────────────────────────────────

hit_rate = retrieval_results["hit"].mean()
mrr = (retrieval_results["rank"].apply(lambda r: 1 / r if r > 0 else 0)).mean()

# Per-language breakdown
lang_stats = (
    retrieval_results.groupby("language")
    .apply(
        lambda g: pd.Series(
            {
                "hit_rate": g["hit"].mean(),
                "mrr": g["rank"].apply(lambda r: 1 / r if r > 0 else 0).mean(),
                "count": len(g),
            }
        ),
        include_groups=False,
    )
    .reset_index()
)

print("=" * 45)
print(" Retrieval Evaluation Results")
print("=" * 45)
print(f"  Hit Rate @5  : {hit_rate:.3f}  ({hit_rate * 100:.1f}%)")
print(f"  MRR          : {mrr:.3f}")
print(f"  Total queries: {len(retrieval_results)}")
print("\nPer-language breakdown:")
print(lang_stats.to_string(index=False))

In [ ]:
# ── Full results table (sorted by rank) ───────────────────────────────────────

print("Retrieval results table (first 20 rows):")
retrieval_results.sort_values("hit", ascending=True).head(20)

---
# Section B: LLM-as-Judge Evaluation

For each ground truth question we:
1. Run the full portfolio agent to get an answer
2. Pass the question + agent answer to a separate Groq judge call
3. The judge scores the answer on four criteria (returned as JSON)

**Judge criteria (each scored per the Pydantic model below):**

| Field | Type | Meaning |
|---|---|---|
| `relevance` | int 1–5 | Is the answer on-topic? |
| `accuracy` | int 1–5 | Is the content factually correct given the expected answer? |
| `language_match` | bool | Did the agent respond in the same language as the question? |
| `source_cited` | bool | Does the response include a GitHub URL as a source? |

In [ ]:
from pydantic import BaseModel
import google.genai as genai
import os


class JudgeResult(BaseModel):
    relevance: int       # 1-5: how relevant is the answer to the question?
    accuracy: int        # 1-5: how factually accurate vs. expected answer?
    language_match: bool # did the agent respond in the same language as the question?
    source_cited: bool   # does the response include a GitHub URL?


JUDGE_PROMPT_TEMPLATE = """\
You are an expert evaluator for a bilingual AI assistant that answers questions about a portfolio website.

Evaluate the AI's response using the criteria below. Return ONLY a valid JSON object — no markdown, no extra text.

--- QUESTION ---
{question}

--- EXPECTED ANSWER (ground truth) ---
{expected_answer}

--- AI AGENT RESPONSE ---
{agent_response}

--- EVALUATION CRITERIA ---
relevance     (int 1-5): Is the agent response relevant to the question?
accuracy      (int 1-5): Is the factual content correct compared to the expected answer?
language_match (bool)  : Did the agent respond in the SAME language as the question?
                         (question language: {language})
source_cited  (bool)   : Does the response contain a GitHub URL (github.com)?

Scoring guide for int fields:
  5 = excellent, 4 = good, 3 = acceptable, 2 = poor, 1 = wrong/irrelevant

Respond with ONLY this JSON:
{{"relevance": <int>, "accuracy": <int>, "language_match": <bool>, "source_cited": <bool>}}
"""

print("JudgeResult model and JUDGE_PROMPT_TEMPLATE defined.")

In [ ]:
# Gemini client for the judge (uses GEMINI_API_KEY from environment)
judge_client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
JUDGE_MODEL = "gemini-2.5-flash-lite"

# Pause between API calls to stay within free-tier rate limits
CALL_DELAY = 2.0      # seconds between successive API calls
MAX_RETRIES = 4       # max retry attempts on 429
BACKOFF_BASE = 5      # initial wait on 429 (doubles each retry)


def call_judge(question: str, expected_answer: str, agent_response: str, language: str) -> JudgeResult:
    """
    Call the Gemini judge model to score one agent response.
    Retries automatically on HTTP 429 (rate-limit) with exponential backoff.
    Returns a JudgeResult, or a fallback result if all retries fail.
    """
    prompt = JUDGE_PROMPT_TEMPLATE.format(
        question=question,
        expected_answer=expected_answer,
        agent_response=agent_response,
        language=language,
    )

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = judge_client.models.generate_content(
                model=JUDGE_MODEL,
                contents=prompt,
                config=genai.types.GenerateContentConfig(
                    response_mime_type="application/json",
                    temperature=0.0,
                ),
            )
            raw = response.text.strip()

            # Strip any accidental markdown fences
            if raw.startswith("```"):
                raw = raw.split("```")[1]
                if raw.startswith("json"):
                    raw = raw[4:]
                raw = raw.strip()

            data = json.loads(raw)
            return JudgeResult(**data)

        except Exception as e:
            err_str = str(e)
            if "429" in err_str and attempt < MAX_RETRIES:
                wait = BACKOFF_BASE * (2 ** (attempt - 1))   # 5s, 10s, 20s, 40s
                print(f"  [429] Rate limit hit — waiting {wait}s before retry {attempt}/{MAX_RETRIES-1}...")
                time.sleep(wait)
            else:
                print(f"  [WARN] Judge error (attempt {attempt}): {e}")
                # Return neutral fallback so the pipeline continues
                return JudgeResult(relevance=1, accuracy=1, language_match=False, source_cited=False)

    return JudgeResult(relevance=1, accuracy=1, language_match=False, source_cited=False)


print(f"call_judge() ready  (model={JUDGE_MODEL}, delay={CALL_DELAY}s, max_retries={MAX_RETRIES})")

### Run the Agent + Judge Pipeline

For each ground truth row:
1. Call `run_agent(agent, question)` — gets the portfolio agent's response
2. Call `call_judge(...)` — scores the response

A `tqdm` bar tracks progress. Results are collected into a list of dicts.

In [ ]:
async def run_llm_judge_eval(ground_truth_df: pd.DataFrame) -> pd.DataFrame:
    """
    For each row in ground_truth_df:
      1. Run the portfolio agent to get an answer
      2. Wait CALL_DELAY seconds  (avoids rate limits)
      3. Run the judge to score the answer
      4. Wait CALL_DELAY seconds again

    Returns a DataFrame with the original columns plus judge scores.
    """
    eval_rows = []

    for _, row in tqdm(ground_truth_df.iterrows(), total=len(ground_truth_df), desc="LLM-as-Judge eval"):

        # 1. Get agent response (async — uses Gemini internally)
        try:
            agent_response = await run_agent(agent, row["question"])
        except Exception as e:
            print(f"  [WARN] Agent error for {row['question_id']}: {e}")
            agent_response = ""

        # 2. Pause before judge call
        await asyncio.sleep(CALL_DELAY)

        # 3. Judge the response (sync Gemini call, retry on 429 built-in)
        judge = call_judge(
            question=row["question"],
            expected_answer=row["expected_answer"],
            agent_response=agent_response,
            language=row["language"],
        )

        # 4. Pause before next agent call
        await asyncio.sleep(CALL_DELAY)

        eval_rows.append(
            {
                "question_id": row["question_id"],
                "question": row["question"],
                "language": row["language"],
                "doc_id": row["doc_id"],
                "agent_response": agent_response,
                "relevance": judge.relevance,
                "accuracy": judge.accuracy,
                "language_match": judge.language_match,
                "source_cited": judge.source_cited,
            }
        )

    return pd.DataFrame(eval_rows)


# ── Run the evaluation ────────────────────────────────────────────────────────
# asyncio.run() works in a script; Jupyter needs nest_asyncio because its
# event loop is already running.
try:
    import nest_asyncio
    nest_asyncio.apply()
    loop = asyncio.get_event_loop()
    judge_results = loop.run_until_complete(run_llm_judge_eval(ground_truth))
except ImportError:
    judge_results = asyncio.run(run_llm_judge_eval(ground_truth))

print(f"\nLLM-as-Judge evaluation complete: {len(judge_results)} rows")

In [ ]:
# Preview the judge results table
judge_results[["question_id", "language", "relevance", "accuracy", "language_match", "source_cited"]].head(10)

---
# Section C: Summary

Aggregate all metrics into a single summary table.

In [ ]:
# ── Aggregate LLM-as-Judge metrics ────────────────────────────────────────────

avg_relevance = judge_results["relevance"].mean()
avg_accuracy = judge_results["accuracy"].mean()
language_match_rate = judge_results["language_match"].mean()
source_citation_rate = judge_results["source_cited"].mean()

print("=" * 50)
print(" Portfolio AI Agent — Evaluation Summary")
print("=" * 50)
print()
print("Retrieval Metrics (Search Engine)")
print("-" * 50)
print(f"  Hit Rate @5          : {hit_rate:.3f}  ({hit_rate * 100:.1f}%)")
print(f"  MRR                  : {mrr:.3f}")
print()
print("LLM Quality Metrics (Agent Responses)")
print("-" * 50)
print(f"  Avg Relevance (1-5)  : {avg_relevance:.2f}")
print(f"  Avg Accuracy  (1-5)  : {avg_accuracy:.2f}")
print(f"  Language Match Rate  : {language_match_rate:.3f}  ({language_match_rate * 100:.1f}%)")
print(f"  Source Citation Rate : {source_citation_rate:.3f}  ({source_citation_rate * 100:.1f}%)")
print()
print(f"  Total questions evaluated: {len(judge_results)}")
print("=" * 50)

In [ ]:
# ── Per-language breakdown ────────────────────────────────────────────────────

# Merge judge results with retrieval results on question_id
merged = judge_results.merge(
    retrieval_results[["question_id", "hit", "rank"]],
    on="question_id",
    how="left",
)
merged["reciprocal_rank"] = merged["rank"].apply(lambda r: 1 / r if r > 0 else 0)

per_lang = (
    merged.groupby("language")
    .agg(
        count=("question_id", "count"),
        hit_rate=("hit", "mean"),
        mrr=("reciprocal_rank", "mean"),
        avg_relevance=("relevance", "mean"),
        avg_accuracy=("accuracy", "mean"),
        language_match_rate=("language_match", "mean"),
        source_citation_rate=("source_cited", "mean"),
    )
    .round(3)
    .reset_index()
)

print("Per-language breakdown:")
per_lang

In [ ]:
# ── Print a markdown-ready summary table ─────────────────────────────────────

summary_md = f"""
## Evaluations

Evaluated on {len(judge_results)} bilingual Q&A pairs ({(ground_truth['language']=='es').sum()} ES / {(ground_truth['language']=='en').sum()} EN).

### Retrieval (Search Engine)

| Metric | Score |
|---|---|
| Hit Rate @5 | {hit_rate:.3f} |
| MRR | {mrr:.3f} |

### LLM Quality (Agent Responses)

| Metric | Score |
|---|---|
| Avg Relevance (1–5) | {avg_relevance:.2f} |
| Avg Accuracy (1–5) | {avg_accuracy:.2f} |
| Language Match Rate | {language_match_rate:.1%} |
| Source Citation Rate | {source_citation_rate:.1%} |
"""

print(summary_md)
print("\n" + "=" * 60)
print("Copy the table above into your README under ## Evaluations")
print("=" * 60)

In [ ]:
# ── Save detailed results to CSV for further analysis ─────────────────────────

RESULTS_PATH = os.path.join(NOTEBOOK_DIR, "eval_results.csv")
merged.to_csv(RESULTS_PATH, index=False, encoding="utf-8")
print(f"Detailed results saved to: {RESULTS_PATH}")

---
## Done!

All evaluation metrics have been computed.

**Files written:**
- `eval/eval_results.csv` — per-question detailed scores

**Next steps:**
- Copy the markdown table from cell above into your `README.md` under `## Evaluations`
- If Hit Rate or Accuracy is low, consider:
  - Tuning `SEARCH_TOP_K` or boost weights in `SearchTool`
  - Adjusting chunk `window_size` / `step` in `ingest.py`
  - Refining the system prompt in `search_agent.py`